In [ ]:
import time
import json
import re
import pandas as pd
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager

# 2026-07-29 업데이트: 올리브영 사이트 개편 대응 재작성

기존 크롤러는 올리브영이 사이트를 전면 개편(카테고리 ID 재발급, 상품 상세페이지를 Shadow DOM 기반 웹 컴포넌트로 전환)하면서 대부분의 셀렉터가 깨져 있었습니다. 아래 셀들을 최신 구조에 맞게 다시 작성했습니다.

**바뀐 점**
- 카테고리 ID(`dispCatNo`)가 전부 재발급되어 새 값으로 교체
- 상품 상세페이지(성분/리뷰)는 여러 겹의 Shadow DOM(`oy-review-*` 웹 컴포넌트)으로 감싸져 있어, 모든 Shadow Root를 재귀적으로 탐색하는 `deepQueryAll`을 JS로 구현해서 사용
- 리뷰는 페이지네이션이 아니라 무한 스크롤 방식으로 로드됨
- 예전의 보습력/끈적임/향 같은 카테고리별 커스텀 평가 항목은 사라지고, 대신 피부타입/톤/피부고민 등 리뷰어가 선택한 `skin_types` 태그로 대체됨

**알려진 한계**
- 별점(`rating`)은 별 아이콘 SVG의 `stroke` 색상(`#FF5753` = 채워진 별)으로 판별합니다. 개발 중 확인한 리뷰가 대부분 5점이라 4점 이하 리뷰로는 검증하지 못했으니, 실사용 전에 평점 낮은 상품으로 한 번 더 확인하는 것을 권장합니다.
- 기본값(`max_products`, `max_reviews_per_product`)을 작게 잡아뒀습니다. 예전처럼 상품 전체 + 리뷰 전체를 한 번에 긁으면 상품 하나당 리뷰가 수만~수십만 건이라 몇 시간이 걸릴 수 있습니다. 필요한 만큼만 점진적으로 늘려서 사용하세요.
- 봇 차단(Cloudflare) 회피를 위해 반드시 메인페이지를 먼저 방문(`bootstrap_session`)한 뒤 상세페이지로 이동해야 합니다.


In [ ]:
# 2025~2026 사이트 개편으로 카테고리 ID(dispCatNo)가 전부 재발급되었습니다.
# 아래 값은 메인 메뉴 > 카테고리 를 직접 눌러서 확인한 최신 ID입니다 (2026-07-29 기준).
# 예전에는 bodyLotion / bodyCream 이 분리되어 있었지만 지금은 "바디로션/크림"으로 통합되었습니다.
MAIN_URL = "https://www.oliveyoung.co.kr/store/main/main.do"

CATEGORY_IDS = {
    "bodyLotionCream": "100000100030025",  # 바디케어 > 바디로션/크림 (구 bodyLotion + bodyCream)
    "bodyOilMist": "100000100030026",      # 바디케어 > 오일/미스트 (구 bodyOil, 미확인 시 카테고리 메뉴에서 재확인 필요)
    "handCare": "100000100030016",         # 바디케어 > 핸드케어 (구 handCream)
    "lipMakeup": "100000100020006",        # 메이크업 > 립메이크업 (구 lipCare와 완전히 동일 카테고리는 아닐 수 있음)
    "skinToner": "100000100010013",        # 스킨케어 > 스킨/토너
    "essenceSerum": "1000001000100140001",  # 스킨케어 > 에센스/세럼/앰플 (2026-07-29 실측: 936개 등록)
    "cream": "1000001000100150001",         # 스킨케어 > 크림 (2026-07-29 실측: 702개 등록, 젤크림 텍스처 포함)
    "mistOil": "100000100010010",           # 스킨케어 > 미스트/오일 (2026-07-29 실측: 157개 등록)
}

def category_url(disp_cat_no, page_idx=1, rows_per_page=24):
    """카테고리 상품 목록 페이지 URL. 이 목록 페이지 자체는 예전 구조 그대로라 pageIdx로 페이지네이션 가능.
    prdSort 코드도 사이트 개편으로 재배정됨: 01=인기순, 02=신상품순, 03=판매순 (2026-07-29 기준 확인).
    구 코드(예전 02=판매순)를 그대로 쓰면 리뷰가 하나도 없는 신상품만 계속 걸리니 주의."""
    return (
        "https://www.oliveyoung.co.kr/store/display/getMCategoryList.do"
        f"?dispCatNo={disp_cat_no}&prdSort=01&pageIdx={page_idx}&rowsPerPage={rows_per_page}"
    )

def detail_url(goods_no, disp_cat_no):
    return f"https://www.oliveyoung.co.kr/store/goods/getGoodsDetail.do?goodsNo={goods_no}&dispCatNo={disp_cat_no}"

In [ ]:
def setup_driver():
    """Chrome WebDriver 설정 (봇 탐지 우회 옵션 포함)"""
    options = Options()
    # options.add_argument('--headless')  # 헤드리스로 돌리면 봇 차단 확률이 올라갈 수 있어 기본은 headful
    options.add_argument('--window-size=1920,1400')
    options.add_argument("--disable-blink-features=AutomationControlled")
    options.add_experimental_option("excludeSwitches", ["enable-automation"])
    options.add_experimental_option('useAutomationExtension', False)
    options.add_argument(
        'user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 '
        '(KHTML, like Gecko) Chrome/118.0.0.0 Safari/537.36'
    )
    service = Service(ChromeDriverManager().install())
    driver = webdriver.Chrome(service=service, options=options)
    driver.execute_script("Object.defineProperty(navigator, 'webdriver', {get: () => undefined})")
    return driver


def bootstrap_session(driver):
    """카테고리/상세 페이지로 바로 진입하면 Cloudflare 사람인증 화면이 뜬다.
    메인페이지를 먼저 방문해서 정상 세션 쿠키를 확보한 뒤 이동해야 한다."""
    driver.get(MAIN_URL)
    time.sleep(3)


# 상품 상세페이지의 리뷰 영역은 oy-review-* 커스텀 엘리먼트들이 서로 중첩된 Shadow DOM 구조라
# 일반 querySelector로는 안까지 들어갈 수 없다. 모든 Shadow Root를 재귀적으로 훑는 헬퍼.
_DEEP_QUERY_JS = """
function deepQueryAll(selector, root) {
    root = root || document;
    const scanRoot = root.shadowRoot ? root.shadowRoot : root;
    let results = Array.from(scanRoot.querySelectorAll(selector));
    const all = scanRoot.querySelectorAll('*');
    for (const el of all) {
        if (el.shadowRoot) results = results.concat(deepQueryAll(selector, el));
    }
    return results;
}
"""

_CLICK_TEXT_JS = _DEEP_QUERY_JS + """
const needle = arguments[0];
const all = document.querySelectorAll('body *');
for (const el of all) {
    let text = '';
    for (const node of el.childNodes) {
        if (node.nodeType === Node.TEXT_NODE) text += node.textContent;
    }
    text = text.trim();
    if (text === needle) {
        el.scrollIntoView({block: 'center'});
        el.click();
        return true;
    }
}
return false;
"""

# "상품정보 제공고시" 표(화장품법 표시 의무사항) 안에서 전성분 행 + 제조사/책임판매업자 행을 함께 반환
_GET_DISCLOSURE_JS = """
function findRowValue(label) {
    const rows = document.querySelectorAll('body *');
    for (const el of rows) {
        if (el.children.length === 0 && el.textContent.trim() === label) {
            let row = el.closest('tr') || el.parentElement.parentElement;
            if (row) {
                const valueCell = row.querySelector('td') || row.lastElementChild;
                if (valueCell) return valueCell.textContent.trim();
            }
        }
    }
    return null;
}
return {
    ingredient: findRowValue('화장품법에 따라 기재해야 하는 모든 성분'),
    manufacturer_raw: findRowValue('화장품제조업자,화장품책임판매업자 및 맞춤형화장품판매업자')
};
"""

# 리뷰 하나(oy-review-review-item)마다 필요한 필드를 Shadow DOM을 뚫고 추출
_EXTRACT_REVIEWS_JS = _DEEP_QUERY_JS + """
const items = deepQueryAll('oy-review-review-item');
return items.map(item => {
    const nameEl = deepQueryAll('.name', item)[0];
    const dateEl = deepQueryAll('.date', item)[0];
    const contentEl = deepQueryAll('.content p', item)[0] || deepQueryAll('.content', item)[0];
    const optionEl = deepQueryAll('.goods-option', item)[0];
    const skinTypeEls = deepQueryAll('.skin-type', item);
    const tagEls = deepQueryAll('.tag', item);
    const starIcons = deepQueryAll('oy-review-star-icon', item);

    const starColors = starIcons.map(star => {
        const svg = star.shadowRoot ? star.shadowRoot.querySelector('svg') : null;
        const path = svg ? svg.querySelector('path') : null;
        return path ? path.getAttribute('stroke') : null;
    });
    // 채워진 별 색상은 #FF5753. (개발 중 5점 리뷰로만 확인했음 - 4점 이하 케이스 재검증 권장)
    const filledCount = starColors.filter(c => c === '#FF5753').length;

    return {
        name: nameEl ? nameEl.textContent.trim() : null,
        date: dateEl ? dateEl.textContent.trim() : null,
        review: contentEl ? contentEl.textContent.trim() : null,
        option: optionEl ? optionEl.textContent.trim() : null,
        skin_types: skinTypeEls.map(e => e.textContent.trim()),
        tags: tagEls.map(e => e.textContent.trim()),
        rating: filledCount,
    };
});
"""


def get_category_products(driver, disp_cat_no, max_products=10, rows_per_page=24):
    """카테고리 목록 페이지(예전 구조 그대로 살아있음)에서 상품 기본 정보만 수집"""
    driver.get(category_url(disp_cat_no, page_idx=1, rows_per_page=rows_per_page))
    time.sleep(2)
    products = []
    cards = driver.find_elements(By.CSS_SELECTOR, "ul.cate_prd_list li")
    for card in cards:
        try:
            anchor = card.find_element(By.CSS_SELECTOR, "a.prd_thumb")
            goods_no = anchor.get_attribute("data-ref-goodsno")
            brand = card.find_element(By.CLASS_NAME, "tx_brand").text.strip()
            name = card.find_element(By.CLASS_NAME, "tx_name").text.strip()
            if goods_no:
                products.append({"goods_no": goods_no, "brand_name": brand, "product_name": name})
        except Exception:
            continue
        if len(products) >= max_products:
            break
    return products


def get_product_disclosure(driver, goods_no, disp_cat_no):
    """상품설명 탭 > 더보기 > 상품정보 제공고시를 펼쳐서 전성분 + 제조사 원문을 함께 가져온다"""
    driver.get(detail_url(goods_no, disp_cat_no))
    time.sleep(3)
    try:
        more_btn = driver.find_element(
            By.CSS_SELECTOR,
            "section.GoodsDetailTabs_product-info-panel__RuH4U button.GoodsDetailTabs_btn-more__zrJGJ",
        )
        driver.execute_script("arguments[0].scrollIntoView({block:'center'}); arguments[0].click();", more_btn)
        time.sleep(1.5)
    except Exception:
        pass  # 이미 펼쳐져 있거나 상품에 따라 버튼이 없을 수 있음

    driver.execute_script(_CLICK_TEXT_JS, "상품정보 제공고시")
    time.sleep(1.5)
    return driver.execute_script(_GET_DISCLOSURE_JS)


# 라벨 표기 형식이 상품마다 다름:
#   "제조사 / 책임판매업자"                              (단순)
#   "제조 : 제조사 / 책임판매 : 책임판매업자"              (라벨 부착)
#   "품목 - 제조사, 품목 - 제조사, ... / 책임판매업자"     (구성품 여러 개, 제조사도 여러 개)
#   "[품목명] 화장품제조업자 : A / 화장품책임판매업자 : B, [품목명] 화장품제조업자 : C / ..." (구성품별 라벨+괄호 혼합)
#   "병풀 추출물 150ml 2개입: (주)바이온셀 / (주)원씽" (라벨 없이 품목명+콜론이 제조사 자리에 섞여 들어간 경우)
# 라벨(제조/책임판매 등)이 있으면 그 값을 우선 신뢰하고, 없으면 "/" 앞뒤 또는 "-" 뒤 값으로 판단한다.
# 어느 경로든 최종 값 안에 콜론이 남아있으면(품목명이 섞여 들어간 경우) 마지막 콜론 뒤쪽을 실제 회사명으로 본다.
_LABELED_MANU_RE = re.compile(r'(?:화장품)?제조(?:업자|원|사)?\s*[:：]\s*([^\[/,]+)')
_LABELED_BRAND_RE = re.compile(r'(?:화장품)?책임판매(?:업자|원)?\s*[:：]\s*([^\[/,]+)')
_DASH_PAIR_RE = re.compile(r'-\s*([^,/]+)')


def _clean_captured_value(value):
    """캡처된 값 안에 콜론이 남아있으면(품목명이 섞여 들어간 경우) 마지막 콜론 뒤쪽만 취한다."""
    value = value.strip()
    if ":" in value or "：" in value:
        value = re.split(r"[:：]", value)[-1].strip()
    return value


def parse_manufacturer_field(raw):
    """상품정보 제공고시 원문에서 manufacturer(제조사)와 brand(책임판매업자)를 분리한다.

    BARE fit 스키마 기준: `manufacturer`=실제 생산 공장, `brand`=책임판매업자.
    둘 다 명확히 뽑히지 않으면 단정하지 않고 manufacturer_confidence를 '확인 필요'로 낮춘다
    (도메인 규칙: 미검증 브랜드-제조사 매핑은 사실로 취급하지 않는다).
    """
    if not raw:
        return {"manufacturer": "", "brand": "", "manufacturer_confidence": "확인 필요"}

    labeled_manus = [_clean_captured_value(m) for m in _LABELED_MANU_RE.findall(raw)]
    labeled_brands = [_clean_captured_value(b) for b in _LABELED_BRAND_RE.findall(raw)]

    if labeled_manus or labeled_brands:
        manufacturers = sorted(set(m for m in labeled_manus if m))
        brand = ", ".join(sorted(set(b for b in labeled_brands if b)))
    else:
        parts = raw.split("/")
        manu_part = parts[0]
        brand = _clean_captured_value(parts[-1]) if len(parts) >= 2 else ""
        pairs = _DASH_PAIR_RE.findall(manu_part)
        if pairs:
            manufacturers = sorted(set(_clean_captured_value(p) for p in pairs if p.strip()))
        else:
            cleaned = _clean_captured_value(manu_part)
            manufacturers = [cleaned] if cleaned else []

    manufacturer_str = ", ".join(manufacturers)
    confidence = "high" if (manufacturer_str and brand) else "확인 필요"
    return {
        "manufacturer": manufacturer_str,
        "brand": brand,
        "manufacturer_confidence": confidence,
    }


def get_product_reviews(driver, goods_no, disp_cat_no, max_reviews=20, max_scroll_rounds=15):
    """리뷰 탭은 무한 스크롤 방식이라, 목표 개수에 도달하거나 더 이상 늘지 않을 때까지 스크롤한다"""
    driver.get(detail_url(goods_no, disp_cat_no) + "&tab=review")
    time.sleep(4)

    tabs = driver.find_elements(By.CSS_SELECTOR, "button.GoodsDetailTabs_tab-item__tgAnU")
    for t in tabs:
        if "리뷰" in t.text:
            driver.execute_script("arguments[0].scrollIntoView({block:'center'}); arguments[0].click();", t)
            break
    time.sleep(5)  # 리뷰 위젯이 비동기로 로드되므로 충분히 대기

    prev_count = -1
    for _ in range(max_scroll_rounds):
        reviews = driver.execute_script(_EXTRACT_REVIEWS_JS)
        if len(reviews) >= max_reviews or len(reviews) == prev_count:
            break
        prev_count = len(reviews)
        driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
        time.sleep(1.5)

    reviews = driver.execute_script(_EXTRACT_REVIEWS_JS)
    return reviews[:max_reviews]


def crawling(disp_cat_no, category_key, max_products=5, max_reviews_per_product=30):
    """카테고리 하나를 대상으로 상품 -> 제조사/성분 -> 리뷰 순으로 수집한다.

    BARE fit 스키마에 맞춰 상품 정보(products)와 리뷰(reviews)를 분리해서 반환한다.
    - products: 상품 1건당 1행. STEP1(스캔)+STEP2(제조사 DB)에 해당하는 정보.
      크롤링으로 얻은 값이라 실제 카메라 OCR과는 출처가 다르므로 data_source에 명시한다.
    - reviews: 리뷰 1건당 1행. product_id로 products와 조인한다.

    주의: 인기 상품은 리뷰가 수십만 건이라 max_reviews_per_product를 크게 잡으면
    상품 1개당 스크롤 라운드가 수천 번씩 필요할 수 있다. 필요한 만큼만 늘릴 것.
    """
    driver = setup_driver()
    products_out = []
    reviews_out = []
    try:
        bootstrap_session(driver)
        products = get_category_products(driver, disp_cat_no, max_products=max_products)
        print(f"[{disp_cat_no}] {len(products)}개 상품 수집 대상 (max_products={max_products})")

        for idx, p in enumerate(products, start=1):
            product_id = f"{category_key}_{idx:02d}"
            print("  처리 중:", p["brand_name"], p["product_name"])
            disclosure = get_product_disclosure(driver, p["goods_no"], disp_cat_no)
            manu = parse_manufacturer_field(disclosure.get("manufacturer_raw"))
            time.sleep(1)
            reviews = get_product_reviews(driver, p["goods_no"], disp_cat_no, max_reviews=max_reviews_per_product)
            print(
                f"    성분 수집: {disclosure.get('ingredient') is not None}, "
                f"제조사: {manu['manufacturer'] or '확인 필요'} (confidence={manu['manufacturer_confidence']}), "
                f"리뷰 수집: {len(reviews)}건"
            )

            products_out.append({
                "product_id": product_id,
                "category": category_key,
                "goods_no": p["goods_no"],
                "product_name": p["product_name"],
                "brand_name": p["brand_name"],
                "manufacturer": manu["manufacturer"],
                "brand": manu["brand"],
                "manufacturer_confidence": manu["manufacturer_confidence"],
                "ingredient": disclosure.get("ingredient"),
                "data_source": "olive_crwl (상품정보 제공고시 크롤링, OCR 아님)",
            })
            for r in reviews:
                reviews_out.append({
                    "product_id": product_id,
                    "customer_id": r["name"],
                    "review": r["review"],
                    "rating": r["rating"],
                    "purchase_date": r["date"],
                    "option": r["option"],
                    "skin_types": ", ".join(r["skin_types"]),
                    "tags": ", ".join(r["tags"]),
                })
            time.sleep(1)  # 상품 간 최소한의 텀을 둬서 과도한 연속 요청을 피한다
    finally:
        driver.quit()
    return {"products": products_out, "reviews": reviews_out}


def build_manufacturer_map(products_df):
    """STEP2 산출물(`manufacturer_map`): 같은 제조사(공장)를 기준으로 브랜드를 묶는다.

    manufacturer_confidence가 'high'인 상품만 신뢰할 수 있는 매핑으로 취급한다(도메인 규칙:
    미검증 브랜드-제조사 매핑은 사실로 취급하지 않음). has_alternative는 같은 제조사를 공유하는
    서로 다른 브랜드가 2개 이상 모였을 때만 True가 되며, FEATURE 02(같은 제조사 탐색)에서
    "대안 제품이 실제로 존재하는지"를 판단하는 데 쓰인다.
    """
    verified = products_df[products_df["manufacturer_confidence"] == "high"]
    manufacturer_map = []
    for manufacturer, group in verified.groupby("manufacturer"):
        if not manufacturer:
            continue
        products = group[["product_id", "product_name", "brand_name", "brand", "category"]].to_dict("records")
        manufacturer_map.append({
            "manufacturer": manufacturer,
            "manufacturer_confidence": "high",
            "brand_count": int(group["brand"].nunique()),
            "has_alternative": bool(group["brand"].nunique() >= 2),
            "products": products,
        })
    unverified_count = int((products_df["manufacturer_confidence"] != "high").sum())
    return {"manufacturer_map": manufacturer_map, "unverified_product_count": unverified_count}

## 2026-07-29 추가 업데이트: BARE fit 스키마 정렬

기존에는 상품 정보(성분/제조사)가 리뷰마다 중복 저장되는 평평한(flat) 구조였습니다. BARE fit의
STEP1(스캔)~STEP2(제조사 DB) 파이프라인에 맞춰 세 가지 산출물로 분리했습니다.

- **products.csv / products.xlsx** — 상품 1건당 1행. `product_id`, `category`, `product_name`,
  `brand_name`(사이트 표시 브랜드명), `manufacturer`(제조사=공장), `brand`(책임판매업자),
  `manufacturer_confidence`, `ingredient`, `data_source`.
  이 데이터는 카메라로 찍은 라벨을 OCR한 것이 아니라 상품 상세페이지의 "상품정보 제공고시"를
  크롤링한 것이므로 `data_source`에 그 출처를 명시합니다(추후 실제 OCR 파이프라인의 `ocr_result`와는
  구분해서 다뤄야 합니다).
- **reviews.csv / reviews.xlsx** — 리뷰 1건당 1행. `product_id`로 products와 연결합니다.
- **manufacturer_map.json** — STEP2 산출물. `manufacturer_confidence`가 `high`인 상품만 모아
  제조사 기준으로 브랜드를 묶습니다. `has_alternative`는 같은 제조사를 공유하는 서로 다른 브랜드가
  2개 이상일 때만 true이며, 이 값이 true인 경우에만 FEATURE 02에서 "같은 공장 대안 제품"을
  제시할 수 있습니다(도메인 규칙: 미검증 매핑을 사실로 단정하지 않음).

현재 샘플(카테고리당 상품 5개, 총 10개)에서는 제조사가 모두 서로 달라 `has_alternative=true`인
그룹이 아직 없습니다. 실제로 같은 공장 매칭 사례를 확인하려면 `max_products`를 늘려 상품 모수를
키워야 합니다.


In [ ]:
CATEGORIES_TO_CRAWL = ["bodyLotionCream", "skinToner"]

all_products = []
all_reviews = []
for key in CATEGORIES_TO_CRAWL:
    result = crawling(CATEGORY_IDS[key], category_key=key, max_products=5, max_reviews_per_product=30)
    all_products.extend(result["products"])
    all_reviews.extend(result["reviews"])

In [ ]:
products_df = pd.DataFrame(all_products)
reviews_df = pd.DataFrame(all_reviews)
manufacturer_map = build_manufacturer_map(products_df)

In [ ]:
products_df.to_csv('products.csv', index=False, encoding='utf-8-sig')
reviews_df.to_csv('reviews.csv', index=False, encoding='utf-8-sig')
with open('manufacturer_map.json', 'w', encoding='utf-8') as f:
    json.dump(manufacturer_map, f, ensure_ascii=False, indent=2)

products_df.to_excel('products.xlsx', index=False, engine='openpyxl')
reviews_df.to_excel('reviews.xlsx', index=False, engine='openpyxl')

In [ ]:
products_df = pd.read_csv('products.csv')
reviews_df = pd.read_csv('reviews.csv')

In [ ]:
products_df

## 다른 카테고리도 같은 방식으로 실행

```python
result = crawling(CATEGORY_IDS["handCare"], category_key="handCare", max_products=5, max_reviews_per_product=30)
all_products.extend(result["products"])
all_reviews.extend(result["reviews"])
# 이후 products_df/reviews_df/manufacturer_map을 다시 만들고 저장하는 셀을 재실행하면 handCare가 합쳐집니다.
```

## 규모를 키우고 싶다면
- `max_products`를 늘리기 전에 `get_category_products`만 따로 호출해서 카테고리에 상품이 몇 개인지(`len(...)`) 먼저 확인하세요.
- `max_reviews_per_product`를 크게 잡을수록 상품 1개당 스크롤 라운드가 늘어나 시간이 선형적으로 늘어납니다. 리뷰가 수만 건 이상인 인기 상품은 100~200건 정도로 제한하는 것을 권장합니다.
- `manufacturer_map`에서 `has_alternative=true`인 그룹(같은 공장을 공유하는 서로 다른 브랜드)을 늘리려면 리뷰 수보다 `max_products`(상품 모수)를 늘리는 쪽이 훨씬 효과적입니다.
- "최근 아이템까지"가 목적이라면, 전체를 다시 긁기보다는 리뷰 정렬을 최신순으로 바꾸고 `purchase_date`가 이전에 수집한 최신 날짜보다 오래된 시점에서 멈추는 방식(증분 크롤링)이 훨씬 효율적입니다. 현재 `get_product_reviews`는 사이트 기본 정렬(유용한 순)을 사용하므로, 최신순으로 바꾸려면 리뷰 탭 안의 정렬 버튼(`텍스트: "최신순"`)을 `_CLICK_TEXT_JS`로 클릭한 뒤 스크롤하도록 확장하면 됩니다.
